# NEXTBUY: Cart Size Prediction Model

## Part 5: ML Model - Regression

This notebook builds a model to predict the average cart size per order.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

## Load Data

In [ ]:
PROCESSED_PATH = Path("processed/")
parquet_file = PROCESSED_PATH / "full_data_engineered.parquet"
fallback_file = Path("full_data_engineered.parquet")

if parquet_file.exists():
    print("Loading from processed folder...")
    full_data = pd.read_parquet(parquet_file)
elif fallback_file.exists():
    print("Loading from current directory...")
    full_data = pd.read_parquet(fallback_file)
else:
    raise FileNotFoundError(
        "Engineered data not found. "
        "Run feature engineering notebook first to create full_data_engineered.parquet"
    )

print(f"Loaded: {full_data.shape}")

## Feature Engineering (No Data Leakage)

**Important:** We only use features available BEFORE an order is placed.
Features derived from the current order (like aisle count) would cause data leakage.

In [ ]:
# Create order-level features using only historical/user-level data
order_features = full_data.groupby('order_id').agg({
    'user_id': 'first',
    'product_id': 'count',  # Target: cart_size
    'order_hour_of_day': 'first',
    'order_dow': 'first',
    'days_since_prior_order': 'first',
    'order_number': 'first',
    # User historical features (available before this order)
    'order_frequency': 'first',
    'avg_basket_size': 'first'
}).reset_index()

# Derived from order timing (available before order)
order_features['is_weekend'] = order_features['order_dow'].isin([0, 6]).astype(int)
order_features['is_morning'] = order_features['order_hour_of_day'].between(6, 11).astype(int)
order_features['is_afternoon'] = order_features['order_hour_of_day'].between(12, 17).astype(int)
order_features['is_evening'] = order_features['order_hour_of_day'].between(18, 23).astype(int)

# User order statistics
user_order_count = full_data.groupby('user_id')['order_id'].nunique().reset_index()
user_order_count.columns = ['user_id', 'user_total_orders']
order_features = order_features.merge(user_order_count, on='user_id', how='left')

# Order sequence ratio
order_features['order_ratio'] = order_features['order_number'] / order_features['user_total_orders']

# Is this a first order?
order_features['is_first_order'] = (order_features['order_number'] == 1).astype(int)

order_features.rename(columns={'product_id': 'cart_size'}, inplace=True)

print(f"Order features shape: {order_features.shape}")
print(f"Features: {list(order_features.columns)}")

## Prepare Data for Modeling

In [ ]:
# Features available BEFORE order (no data leakage)
feature_cols = [
    'order_hour_of_day',      # Time of order
    'order_dow',              # Day of week
    'days_since_prior_order', # Days since last order
    'order_number',           # Order sequence number
    'order_frequency',        # User's avg days between orders
    'avg_basket_size',        # User's historical avg basket size
    'is_weekend',            # Weekend flag
    'is_morning',            # Morning
    'is_afternoon',          # Afternoon
    'is_evening',            # Evening
    'user_total_orders',     # User's total order count
    'order_ratio',           # Current order / total orders
    'is_first_order'         # First order flag
]

model_data = order_features[feature_cols + ['cart_size']].dropna()

X = model_data[feature_cols]
y = model_data['cart_size']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Features: {len(feature_cols)}")

## Train Models

In [ ]:
# Model 1: Linear Regression
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)

print(f"Linear Regression - MAE: {lr_mae:.2f}, RMSE: {lr_rmse:.2f}, R²: {lr_r2:.4f}")

In [ ]:
# Model 2: Random Forest
print("Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print(f"Random Forest - MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}, R²: {rf_r2:.4f}")

In [ ]:
# Model 3: Gradient Boosting
print("Training Gradient Boosting...")
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, y_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
gb_r2 = r2_score(y_test, y_pred_gb)

print(f"Gradient Boosting - MAE: {gb_mae:.2f}, RMSE: {gb_rmse:.2f}, R²: {gb_r2:.4f}")

## Model Comparison

In [ ]:
models = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
mae_scores = [lr_mae, rf_mae, gb_mae]
rmse_scores = [lr_rmse, rf_rmse, gb_rmse]
r2_scores = [lr_r2, rf_r2, gb_r2]

comparison = pd.DataFrame({
    'Model': models,
    'MAE': mae_scores,
    'RMSE': rmse_scores,
    'R²': r2_scores
})

print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(comparison.to_string(index=False))

best_idx = r2_scores.index(max(r2_scores))
print(f"\nBest model: {models[best_idx]} (R² = {r2_scores[best_idx]:.4f})")

## Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['Feature'], importance['Importance'], color='teal')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop features:")
for _, row in importance.iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")

## Prediction Example

In [ ]:
# Example: Predict cart size for a new order
new_order = pd.DataFrame({
    'order_hour_of_day': [10],       # Morning
    'order_dow': [2],                # Tuesday
    'days_since_prior_order': [7],  # 7 days since last order
    'order_number': [5],             # 5th order
    'order_frequency': [7.0],       # User orders every 7 days
    'avg_basket_size': [8.0],       # User's avg basket
    'is_weekend': [0],               # Not weekend
    'is_morning': [1],                # Morning order
    'is_afternoon': [0],
    'is_evening': [0],
    'user_total_orders': [10],       # 10 orders total
    'order_ratio': [0.5],            # 5th of 10 orders
    'is_first_order': [0]
})

prediction = rf_model.predict(new_order)[0]

print("Example Prediction:")
print(f"  User's historical avg basket: 8.0 products")
print(f"  Predicted cart size: {prediction:.1f} products")

## Model export

In [ ]:
import joblib
joblib.dump(lr_model, "model_exports/linear_regression_cart_size_model.joblib")
joblib.dump(rf_model, "model_exports/random_forest_cart_size_model.joblib")
joblib.dump(gb_model, "model_exports/gradient_boosting_cart_size_model.joblib")


## Summary

In [ ]:
print("="*60)
print("CART SIZE PREDICTION MODEL - SUMMARY (No Data Leakage)")
print("="*60)
print(f"""
Objective: Predict cart size using ONLY features available before order

Features ({len(feature_cols)} total):
  - order timing: hour, day, weekend, morning/afternoon/evening
  - user history: avg_basket_size, order_frequency, user_total_orders
  - order sequence: order_number, order_ratio, is_first_order
  - prior order: days_since_prior_order

Best Model: {models[best_idx]}
  - MAE: {mae_scores[best_idx]:.2f} products
  - RMSE: {rmse_scores[best_idx]:.2f}
  - R²: {r2_scores[best_idx]:.4f}

Note: Lower R² is realistic - we're predicting WITHOUT knowing
      what products will be in the cart (that would be data leakage).
""")